In [ ]:
%uv pip install pandas numpy scanpy scrublet

In [ ]:
import sys
sys.path.insert(0, '/home/ubuntu/.local/lib/python3.12/site-packages')

In [ ]:
!curl -L -o dropbox_folder_metadata.zip "https://www.dropbox.com/scl/fo/8d0vi4pjnuf4gij3ll262/ADuPk5vj35q71RtELC8vz-c?rlkey=nz112f59ewp1v0rjd4mhjyd8r&dl=1"

In [ ]:
!curl -L -o dropbox_folder_data.zip "https://www.dropbox.com/scl/fo/7wyp7x2irndqhn2drrc0c/AFAqeJ6K7_a6TqXb95UIc1E?rlkey=qjefasm7oha10f7yg74hpmz9q&dl=1"

In [ ]:
!unzip "*.zip"  

In [ ]:
!for f in *.tar.gz; do tar -xzvf "$f"; done

In [ ]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob
import gc

In [ ]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [ ]:
# study files

In [ ]:
study_name = 'Bi2021'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Kidney'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

In [ ]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()